# 02 - Entrenamiento del modelo de detección de fraude

**Prerequisito:** ejecutar `src/ingestion/load_data.py` primero para generar el dataset merged.

**Entrada:** `data/processed/siniestros_merged.csv` (1000 registros, 58 columnas, merge de las 3 fuentes)

**Salida:** `models/fraud_model.pkl` (modelo entrenado, listo para usar en producción)

Este notebook se ejecuta una sola vez. El archivo generado se sube al repo y la aplicación solo lo carga, nunca reentrena.

## 1. Setup

Si ejecutas esto en **Google Colab**, descomenta el bloque de clonado del repo. Si trabajas en local, ignóralo.

In [ ]:
# --- Solo para Google Colab ---
# !git clone https://github.com/Shermy143/fraudia-claims.git
# %cd fraudia-claims
# !pip install -q xgboost shap scikit-learn pandas numpy joblib openpyxl

In [ ]:
import sys
import os

# Permite importar desde src/ sin instalar el paquete
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from features.build_features import construir_features, obtener_matriz_modelo, FEATURES_MODELO

print('Setup OK')

## 2. Cargar el dataset

Cargamos el dataset merged generado por `load_data.py`. Incluye las 1000 filas de la compañera
enriquecidas con `similitud_narrativa_max` (500 registros con valor, 500 con NaN).
XGBoost maneja los NaN nativamente, no se requiere imputación.

In [ ]:
RUTA_DATASET = '../data/processed/siniestros_merged.csv'

df = pd.read_csv(RUTA_DATASET)
print(f'Registros: {len(df)}')
print(f'Columnas: {len(df.columns)}')
df.head()

In [ ]:
# Verificar la distribución de la etiqueta
distribucion = df['etiqueta_fraude_simulada'].value_counts(normalize=True)
print('Distribución de etiqueta:')
print(f'  Normal (0): {distribucion.get(0, 0):.1%}')
print(f'  Fraude (1): {distribucion.get(1, 0):.1%}')

## 3. Feature engineering

`construir_features()` ya incluye el motor de reglas. Devuelve el DataFrame con todas las columnas originales más las features nuevas y el `score_reglas`.

In [ ]:
df_features = construir_features(df)

# Matriz X (solo features del modelo) y vector y (etiqueta)
X = obtener_matriz_modelo(df_features)
y = df_features['etiqueta_fraude_simulada']

print(f'Forma de X: {X.shape}')
print(f'Features usadas: {len(FEATURES_MODELO)}')
X.head()

## 4. Split train/test

80% entrenamiento, 20% test. `stratify=y` asegura que la proporción de fraude sea la misma en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {len(X_train)} registros')
print(f'Test:  {len(X_test)} registros')
print(f'Fraude en train: {y_train.sum()} ({y_train.mean():.1%})')
print(f'Fraude en test:  {y_test.sum()} ({y_test.mean():.1%})')

## 5. Entrenar XGBoost

**Parámetros clave:**
- `n_estimators=100`: número de árboles. Más árboles → más capacidad pero riesgo de sobreajuste.
- `max_depth=4`: profundidad de cada árbol. Bajo para evitar memorizar.
- `learning_rate=0.1`: cuánto aprende cada árbol nuevo. Estándar.
- `scale_pos_weight`: corrige el desbalance 80/20. Sin este parámetro el modelo aprende a decir 'todo normal'.

In [ ]:
# Calcular el peso de la clase positiva automáticamente
ratio_desbalance = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Ratio de desbalance: {ratio_desbalance:.2f}')

modelo = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=ratio_desbalance,
    eval_metric='auc',
    random_state=42,
)

modelo.fit(X_train, y_train)
print('Modelo entrenado')

## 6. Evaluación rápida

El análisis profundo va en `03_evaluacion_modelo.ipynb`. Aquí solo confirmamos que el modelo aprendió algo razonable antes de guardarlo.

In [ ]:
# Predicciones con umbral por defecto (0.5)
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

print('=== AUC-ROC ===')
print(f'{roc_auc_score(y_test, y_proba):.4f}')

print('\n=== Matriz de confusión ===')
print('              Pred Normal | Pred Fraude')
cm = confusion_matrix(y_test, y_pred)
print(f'Real Normal:  {cm[0][0]:11d} | {cm[0][1]:11d}')
print(f'Real Fraude:  {cm[1][0]:11d} | {cm[1][1]:11d}')

### Umbral ajustado a 0.3

En seguros conviene un umbral bajo: es peor dejar pasar un fraude (falso negativo) que investigar un caso de más (falso positivo). Subir el recall a costa de algo de precision es una decisión de negocio consciente.

In [ ]:
UMBRAL = 0.3
y_pred_umbral = (y_proba >= UMBRAL).astype(int)

print(f'=== Classification Report (umbral={UMBRAL}) ===')
print(classification_report(y_test, y_pred_umbral, target_names=['Normal', 'Fraude']))

## 7. Importancia de features

Vista rápida de qué variables pesan más. SHAP (más detallado) va en el notebook 03.

In [ ]:
importancias = pd.DataFrame({
    'feature': FEATURES_MODELO,
    'importancia': modelo.feature_importances_
}).sort_values('importancia', ascending=False)

importancias

## 8. Guardar el modelo

Este `.pkl` es lo que carga `fraud_model.py` en producción. Subirlo al repo en `models/`.

In [ ]:
RUTA_MODELO = '../models/fraud_model.pkl'
os.makedirs(os.path.dirname(RUTA_MODELO), exist_ok=True)

joblib.dump(modelo, RUTA_MODELO)

tamaño_kb = os.path.getsize(RUTA_MODELO) / 1024
print(f'Modelo guardado en {RUTA_MODELO} ({tamaño_kb:.1f} KB)')

## 9. Prueba de carga

Verifica que el modelo se puede recargar y predice igual. Esto es lo que hará `fraud_model.py` en producción.

In [ ]:
modelo_cargado = joblib.load(RUTA_MODELO)

# Predecir con el modelo recargado y comparar
y_proba_cargado = modelo_cargado.predict_proba(X_test)[:, 1]

diferencia = np.abs(y_proba - y_proba_cargado).max()
print(f'Diferencia máxima entre modelo original y cargado: {diferencia:.10f}')
print('OK - el modelo se cargó correctamente' if diferencia < 1e-6 else 'ERROR')

---

**Próximo paso:** abrir `03_evaluacion_modelo.ipynb` para el análisis profundo con SHAP, curva ROC, análisis de falsos positivos y métricas por umbral.